#   GATS   

\-CODIGOS

In [ ]:
import requests
import pandas as pd

API_KEY = 'QDeawdPuGxoBCPXlZaDffXqMKI9Kllf0g6dqqQCc'

url = f'https://api.fas.usda.gov/api/gats/commodities?api_key={API_KEY}'
r = requests.get(url)
df = pd.DataFrame(r.json())

codigos_res = [
    '020110',  # canales frescas
    '020120',  # cortes con hueso frescos
    '020130',  # deshuesado fresco
    '020210',  # canales congeladas
    '020220',  # cortes con hueso congelados
    '020230',  # deshuesado congelado
    '021020'
]

mask_resto = df['hS10Code'].astype(str).str.startswith(tuple(codigos_res))

res = df[mask_resto]
res = res[~res['hS10Code'].astype(str).str.endswith('0000')]
res = res.drop_duplicates(subset='hS10Code', keep='first')
res = res[~res['commodityDescription'].str.contains('VEAL|BISON', case=False, na=False)]

print(f'Total códigos: {len(res)}')
print(res[['hS10Code', 'commodityName', 'commodityDescription']].to_string())
res['hS10Code'] = res['hS10Code'].astype(str).str.zfill(10)
res[['hS10Code', 'commodityName', 'commodityDescription']].to_csv('HS-PRODU-co.csv', index=False)

# Lista para usar en GATS
gats = res['hS10Code'].tolist()
print(f'\nLista gats: {gats}')

Total códigos: 94
       hS10Code     commodityName                                                                                               commodityDescription
280  0201100090  C/HC BOVN, FR/CH                                            MEAT OF BOVINE ANIMALS, FRESH OR CHILLED, CARCASSES AND HALF CARCASSESS
283  0201100590    C/HC BV,NES,FC                                                         GEN NOTE15 CARCASS/HALF-CARCASS BOVINE NESOI FRESH/CHILLED
285  0201101090  C/HCBV,NES,FC,QT                                                         ADDTL NOTE3 CARCASS/HALF-CARCASS BOVNE NESOI FRESH/CHILLED
287  0201105090  C/HCBV,NES,FC,OQ                   OTHER MEAT OF BOVINE ANIMALS, FRESH OR CHILLED, CARCASSES OR HALF CARCASSES, NOT IN NOTE 15 OR 3
288  0201200200     BVCT,HQBI,PFC                                                 GEN NOTE15 HI-QULTY BEEF CUTS WITH BONE IN PROCESSED FRESH/CHILLED
289  0201200400     BV,BI,NES,PFC                                                   GEN 

In [3]:
import requests
import pandas as pd

API_KEY = 'jLOfTULWelr78pb1AV6pOsUQBLaFr6Bizk1M2V2V'

codigos = pd.read_csv('HS-PRODU-co.csv')['hS10Code'].astype(str).unique().tolist()

# 4 flujos: (endpoint, partner, etiqueta)
flujos = [
    ('censusExports', 'MX', 'USA→MX'),
    ('censusImports', 'MX', 'MX→USA'),
    ('censusExports', 'CA', 'USA→CA'),
    ('censusImports', 'CA', 'CA→USA'),
]

# Año y mes de muestra — solo uno para diagnóstico
YEAR, MES = 2023, 6

resultados = []

for endpoint, partner, label in flujos:
    url = (
        f'https://api.fas.usda.gov/api/gats/{endpoint}'
        f'/partnerCode/{partner}/year/{YEAR}/month/{MES}'
        f'?api_key={API_KEY}'
    )
    print(f'Consultando {label}...')
    try:
        r = requests.get(url, timeout=30)
        if r.status_code == 200 and r.json():
            df = pd.DataFrame(r.json())
            df['hS10Code'] = df['hS10Code'].astype(str)

            # Cuáles de nuestros códigos aparecen en la respuesta
            encontrados = set(df['hS10Code']) & set(codigos)
            sin_datos   = set(codigos) - encontrados

            for cod in codigos:
                resultados.append({
                    'codigo' : cod,
                    'flujo'  : label,
                    'datos'  : cod in encontrados
                })
            print(f'  ✅ {len(encontrados)}/{len(codigos)} códigos con datos')
        else:
            print(f'  ⚠️ status {r.status_code}')
    except Exception as e:
        print(f'  ❌ {e}')

# ── Tabla resumen ──────────────────────────────────────────────────────────────
df_res = pd.DataFrame(resultados)
pivot  = df_res.pivot(index='codigo', columns='flujo', values='datos')
pivot['total_flujos'] = pivot.sum(axis=1)
pivot  = pivot.sort_values('total_flujos', ascending=False)

print('\n── Con datos en los 4 flujos ──')
print(pivot[pivot['total_flujos'] == 4])

print('\n── Sin datos en ningún flujo ──')
print(pivot[pivot['total_flujos'] == 0].index.tolist())

pivot.to_csv('diagnostico_codigos_res.csv')
print('\n✅ Guardado en diagnostico_codigos_res.csv')

Consultando USA→MX...
  ✅ 0/52 códigos con datos
Consultando MX→USA...
  ✅ 0/52 códigos con datos
Consultando USA→CA...
  ✅ 0/52 códigos con datos
Consultando CA→USA...
  ✅ 0/52 códigos con datos

── Con datos en los 4 flujos ──
Empty DataFrame
Columns: [CA→USA, MX→USA, USA→CA, USA→MX, total_flujos]
Index: []

── Sin datos en ningún flujo ──
['201100000', '201100090', '202208000', '202300200', '202300400', '202300600', '202301000', '202302000', '202303000', '202303550', '202304000', '202305000', '202305025', '202305035', '202305045', '202305055', '202305065', '202305075', '202305085', '202305090', '202305091', '202305097', '202306000', '202308000', '202308015', '202206000', '202205090', '202205085', '202200600', '201100590', '201101090', '201105090', '202100000', '202100090', '202100590', '202101090', '202105090', '202200200', '202200400', '202201000', '202205075', '202202000', '202203000', '202203550', '202204000', '202205000', '202205025', '202205035', '202205045', '202205055', '2022

,hS10Code,startDate,endDtate,productType,commodityName,commodityDescription,isAgCommodity,censusUOMId1,censusUOMId2,fasConvertedUOMId,fasNonConvertedUOMId
5782,1007000,196701,None,T,"HORSE/MULE,SLTR",NaN,False,78,78,78,78
5783,1007000020,198901,None,M,SORGHUM SEED,GRAIN SORGHUM SEEDS OF A KIND USED FOR SOWING,False,47,47,70,47


\-Unidades para GATS

In [30]:
import requests
import pandas as pd

API_KEY = 'XbgcpZL0C1IAOE71twGoL1s3goxu3al3JfDl4qgx'

url = f'https://api.fas.usda.gov/api/gats/unitsOfMeasure?api_key={API_KEY}'
r = requests.get(url)

df = pd.DataFrame(r.json())
df[df['unitOfMeasureId'] == 70]


,unitOfMeasureId,unitOfMeasureCode,unitOfMeasureName,unitOfMeasureDescription
70,70,MT,Metric Tons,Metric Tons


\-
 Codigos de paises 

In [ ]:
API_KEY = 'XbgcpZL0C1IAOE71twGoL1s3goxu3al3JfDl4qgx'
# Primero obtén todos los códigos de país disponibles
url_paises = f'https://api.fas.usda.gov/api/gats/countries?api_key={API_KEY}'
r = requests.get(url_paises)
paises = pd.DataFrame(r.json())
# Buscar en tu lista de países si hay algún código de mundo/total
print(paises[paises['countryName'].str.contains('USA|Total|', case=False, na=False)])

# STATSCAN

\- Unidades y Factor

In [ ]:
url_meta = "https://www150.statcan.gc.ca/t1/wds/rest/getCodeSets"
r = requests.get(url_meta)
meta = r.json()['object']
for n in meta['scalar']:
    if n['scalarFactorCode'] == 3:#Aqui lo buscas 
        print(f'\n{n['scalarFactorCode']}\n{n['scalarFactorDescEn']}\n{'-'*20}')
for n in meta['uom']:
    if n['memberUomCode'] == 81:
        print(f'\n{n['memberUomCode']}\n{n['memberUomEn']}\n{'-'*20}')    


3
thousands
--------------------

81
Dollars
--------------------
